In [1]:
import os
import duckdb
from dotenv import load_dotenv ,find_dotenv

# 01 .env 파일을 로드
load_dotenv(find_dotenv())

# 02. minio 연결을 위한 정보를 매핑해주기

endpoint = os.getenv('MINIO_ENDPOINT')
access_key = os.getenv('MINIO_ACCESS_KEY')
secret_key = os.getenv('MINIO_SECRET_KEY')
#use_ssl = os.getenv('MINIO_USE_SSL')

#SSL 여부를 boolean값으로 변경해주면 좋다네?
#이 자체는 boolean 값이 아니고 단순히 text비교를 통해 그 결과가 TRUE 아니면 FALSE로 나오게 될텐데 그 값을 저장하는 방식
use_ssl = os.getenv('MINIO_USE_SSL' , 'False').lower() == 'true'

print (f"연결할 주소: {endpoint}")
print (f"액세스 키 확인: {access_key[:5]}***")



연결할 주소: 192.168.219.101:9000
액세스 키 확인: EN37I***


In [4]:
#1. duckdb 연결 생성
con = duckdb.connect()

# 2. httpfs 확장 프로그램 설치 및 로드 
# (S3나 HTTP 주소에 있는 파일을 읽으려면 이 기능이 꼭 필요합니다)
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")


# 3. DuckDB에게 MinIO 접속 정보를 알려줍니다 (아까 가져온 변수들 활용!)
con.execute(f"SET s3_endpoint='{endpoint}';")
con.execute(f"SET s3_access_key_id='{access_key}';")
con.execute(f"SET s3_secret_access_key='{secret_key}';")


# 4. 필수 설정: MinIO는 'path' 주소 스타일을 사용합니다.
con.execute("SET s3_url_style='path';")


# 5. SSL 사용 여부 설정 (아까 만든 Boolean 값 적용)
con.execute(f"SET s3_use_ssl={'true' if use_ssl else 'false'};")


print("✅ DuckDB의 MinIO 접속 준비 완료!")


✅ DuckDB의 MinIO 접속 준비 완료!


In [5]:
# 1. 이제 진짜로 MinIO 안에 있는 파일을 지목합니다. 
# s3:// 뒤에 [버킷 이름]/[파일명.parquet] 순서로 적어주시면 됩니다.
bucket_name = "petroleum-project" # 예: "raw-data"
file_name = "sample_data.parquet"  # 예: "sales_2024.parquet"
path = f"s3://{bucket_name}/{file_name}"
# 2. DuckDB에게 명령을 내립니다.
# "저기 저 주소(path)에 있는 파케이 파일에서 데이터를 10개만 가져와서 판다스(df)로 보여줘!"
df = con.sql(f"SELECT * FROM '{path}' LIMIT 10").df()
# 3. 결과 확인
display(df)

HTTPException: HTTP Error: Unable to connect to URL "http://192.168.219.101:9000/petroleum-project/sample_data.parquet": 404 (Not Found).

In [3]:
# 1. SQL 확장 기능을 로드합니다.
%load_ext sql
# 2. 결과 표시 설정 (Pandas 형식으로 예쁘게 나오게 함)
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
# 3. [핵심!] 우리가 아까 모든 설정을 마친 'con' 변수를 매직 키워드에 연결합니다.
# 마치 "야, 아까 그 똑똑해진 로봇(con)을 이제부터 %sql 쓸 때마다 불러와!"라고 하는 명령입니다.

%sql duckdb:///:memory:
%sql con

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(_duckdb.ParserException) Parser Error: syntax error at or near "con"

LINE 1: con
        ^
[SQL: con]
(Background on this error at: https://sqlalche.me/e/20/f405)



In [6]:
# 1. 일단 SQL 매직의 기본 연결을 DuckDB로 고정합니다.
%sql duckdb:///:memory:
# 2. [가장 중요!] 이 매직 커맨드 연결 "자체"에 설정을 직접 주입합니다.
# %sql 뒤에 바로 SET 명령어를 붙여서 하나씩 실행해 주세요.
%sql SET s3_endpoint='{{endpoint}}';
%sql SET s3_access_key_id='{{access_key}}';
%sql SET s3_secret_access_key='{{secret_key}}';
%sql SET s3_url_style='path';
%sql SET s3_use_ssl='false';
print("✅ 매직 커맨드용 전용 설정 완료!")

✅ 매직 커맨드용 전용 설정 완료!


In [9]:
%%sql

select *
from 's3://petroleum-project/sample_data.parquet'

,id,date,category,value,status
0,1,2026-01-01,C,72.22,False
1,2,2026-01-02,C,195.53,False
2,3,2026-01-03,A,110.76,True
3,4,2026-01-04,A,132.86,False
4,5,2026-01-05,B,275.36,False
...,...,...,...,...,...
95,96,2026-04-06,A,307.61,False
96,97,2026-04-07,B,488.18,False
97,98,2026-04-08,B,419.66,True
98,99,2026-04-09,A,360.09,False


In [10]:
%sql ROLLBACK;

,Success


In [9]:
%%sql

select *
from 's3://petroleum-project/national_avg/*/*.parquet'
where part_dt >= '20260124'
and prodnm = '휘발유'

,part_dt,PRODCD,PRODNM,PRICE,DIFF,collect_time
0,20260124,B027,휘발유,1692.00,-0.91,2026-01-24 23:07:33
1,20260125,B027,휘발유,1692.02,-0.01,2026-01-25 01:00:10
2,20260126,B027,휘발유,1691.89,+0.08,2026-01-26 01:00:10
3,20260127,B027,휘발유,1691.14,-0.32,2026-01-27 01:00:09
